### Part 1

### What are we doing?

First, load the customer data so we can look at it.

Then check which columns have missing information.

Basically:

**Step 1:** Open the data  
**Step 2:** Find the blanks  
**Step 3:** Decide how to fill them in

In [1]:
import pandas as pd

In [2]:
df = pd.read_excel("../Data/datasets_lab1/customer_churn.xlsx")
df.head()

,customer_id,age,income,education_level,subscription_type,days_since_last_purchase,customer_satisfaction,region,churn
0,1,25.0,45000.0,Bachelor,Premium,30.0,4,North,0
1,2,NaN,65000.0,Master,Basic,45.0,3,South,0
2,3,35.0,55000.0,High School,Premium,15.0,5,East,1
3,4,42.0,75000.0,PhD,Premium,60.0,2,West,1
4,5,28.0,48000.0,Bachelor,Basic,NaN,4,North,0


### “For every column, count how many blank/missing values there are."
### Check for missing stuff

Now we check every column to see where data is missing.

`NaN` basically means:

**"There should be a value here, but there isn't one."**

So when you see "age = NaN", that means customer has no age entered. 

In [3]:
df.isnull().sum()

customer_id                 0
age                         7
income                      2
education_level             0
subscription_type           0
days_since_last_purchase    2
customer_satisfaction       0
region                      0
churn                       0
dtype: int64

### What are we checking now?

We have missing values, but before filling them in, we want to look at the numbers.

We are checking:

- **minimum** = smallest number
- **maximum** = biggest number
- **mean** = average
- **median** = middle number

Why?

Because the **mean can get messed up by really big or really small values**.

The **median is usually safer when the data has weird/extreme values**.

### Why are we doing this?

We need to decide how to fill the blanks.

So we compare:

- average
- middle value
- smallest number
- biggest number

If the numbers are kinda wild, median is usually the safer choice.

Before we decide mean vs median, let’s actually look at the numbers instead of guessing. That will show us things like:

minimum
maximum
average (mean)
middle value (50% = median)

In [4]:
df[['age', 'income', 'days_since_last_purchase']].describe()

,age,income,days_since_last_purchase
count,38.000000,43.000000,43.000000
mean,36.684211,60139.534884,60.697674
std,8.249833,13647.922213,37.107011
min,24.000000,41000.000000,15.000000
25%,30.000000,47500.000000,26.500000
50%,35.500000,58000.000000,45.000000
75%,42.000000,72500.000000,92.500000
max,55.000000,85000.000000,130.000000


Numbers Tell us:
AGE
mean   = 36.68
median = 35.50
max    = 55

INCOME
mean   = 60,139
median = 58,000
max    = 85,000

DAYS SINCE LAST PURCHASE
mean   = 60.70
median = 45
max    = 130

Quick translation: 50% in that table = median.

### What do these numbers tell me?

We are comparing the **average** to the **middle value**.

- **Age:** average = 36.68, middle = 35.5
- **Income:** average = 60,139, middle = 58,000
- **Days since last purchase:** average = 60.7, middle = 45

The bigger the gap between the average and the middle, the more the data may be pulled around by high or low values.

The biggest difference is in **days since last purchase**, so the median is safer there.

Quick reminder:

- **mean = average**
- **median = middle**
- **50% in the table = median**

### Part 1: Handling Missing Values

**Question 1a:** I would use the median for age because the median is less affected by outliers or extreme values than the mean.

**Question 1b:** I would use the median for income because income values can be skewed by unusually high or low incomes.

**Question 1c:** I would use the median for days since last purchase because the data is skewed. The mean is about 60.7 days while the median is 45 days, so the median is less affected by large values.

### Now Fill in Missing Values:

In [5]:
df['age'] = df['age'].fillna(df['age'].median())
df['income'] = df['income'].fillna(df['income'].median())
df['days_since_last_purchase'] = df['days_since_last_purchase'].fillna(
    df['days_since_last_purchase'].median()
)

### Show it worked:

In [6]:
df.isnull().sum()

customer_id                 0
age                         0
income                      0
education_level             0
subscription_type           0
days_since_last_purchase    0
customer_satisfaction       0
region                      0
churn                       0
dtype: int64

### Part 2: Education Encoding

It now shows 0 missing values in every column

### Part 2: Ordinal Encoding

**Question 2:** Education level is suitable for ordinal encoding because the categories have a meaningful order. High School comes before Bachelor, Bachelor before Master, and Master before PhD.

In [7]:
education_map = {
    'High School': 1,
    'Bachelor': 2,
    'Master': 3,
    'PhD': 4
}

df['education_encoded'] = df['education_level'].map(education_map)

One more cell to check it:

In [8]:
df[['education_level', 'education_encoded']].head(10)

,education_level,education_encoded
0,Bachelor,2
1,Master,3
2,High School,1
3,PhD,4
4,Bachelor,2
5,Master,3
6,High School,1
7,Bachelor,2
8,High School,1
9,Master,3


### What is happening here?

We made a number version of the education level.

- High School = 1
- Bachelor = 2
- Master = 3
- PhD = 4

So the original `education_level` column stays there, and the new `education_encoded` column shows the number version.

Example:

- Bachelor → 2
- Master → 3
- High School → 1
- PhD → 4

We can do this because education levels have a real order from lower to higher.

### Part 3: One-Hot Encoding

**Question 3a:** Region is better suited for one-hot encoding because the regions do not have a meaningful order.

**Question 3b:** Four new columns will be created because there are four regions: North, South, East, and West.

**Question 3c:** Ordinal encoding would incorrectly make the regions look ranked, such as West being greater than North, even though the regions have no ranking.

Dumbed Down: works = numbers, but keeping the correct order

In [9]:
region_dummies = pd.get_dummies(df['region'], prefix='region')

region_dummies.head()

,region_East,region_North,region_South,region_West
0,False,True,False,False
1,False,False,True,False
2,True,False,False,False
3,False,False,False,True
4,False,True,False,False


### What this output shows

The `region` column has been converted into four separate columns:

- `region_East`
- `region_North`
- `region_South`
- `region_West`

Each row uses `True` or `False` to show which region that customer belongs to.

For example:

- Row 0 has `region_North = True`, so that customer is from the North region.
- Row 1 has `region_South = True`, so that customer is from the South region.
- Row 2 has `region_East = True`.
- Row 3 has `region_West = True`.

Only one region should be `True` for each customer because each customer belongs to one region.

This is useful for machine learning because the regions are now represented as separate numeric-style categories without creating a fake ranking between them.

### What this means

The old `region` column had words like North, South, East, and West.

Machine learning likes numbers better than words.

So we turned each region into its own column.

- `True` = yes, the customer is from that region
- `False` = no, the customer is not from that region

Example:
- If `region_North = True`, that customer is from North.
- The other region columns for that same customer will be `False`.

This is better than giving the regions numbers like 1, 2, 3, 4 because that would make it look like the regions have an order or ranking, which they do not.

Before:
North

After:
North = yes
South = no
East = no
West = no

### 1. Add New Columns Back Onto df
### 2. Delete the Old Region Column

In [10]:
df = pd.concat([df, region_dummies], axis=1)
df = df.drop('region', axis=1)

final check:

In [11]:
df.head()

,customer_id,age,income,education_level,subscription_type,days_since_last_purchase,customer_satisfaction,churn,education_encoded,region_East,region_North,region_South,region_West
0,1,25.0,45000.0,Bachelor,Premium,30.0,4,0,2,False,True,False,False
1,2,35.5,65000.0,Master,Basic,45.0,3,0,3,False,False,True,False
2,3,35.0,55000.0,High School,Premium,15.0,5,1,1,True,False,False,False
3,4,42.0,75000.0,PhD,Premium,60.0,2,1,4,False,False,False,True
4,5,28.0,48000.0,Bachelor,Basic,45.0,4,0,2,False,True,False,False


### Final Check - What This Means

The data is now cleaned up and easier for machine learning to use.

- Missing values were filled in.
- Education was turned into numbers.
- Region was split into separate True/False columns.
- The old `region` column was removed.

So basically, we took messy human-readable data and changed it into a cleaner format that a machine learning model can understand better.

Before: blanks + words      
After: no blanks + more numbers/True-False columns     
Result: data is more ML-ready.